# Visualization 1

## Cleaning Data

In [1]:
import pandas as pd
import altair as alt

alt.data_transformers.enable('json')

data = pd.read_csv('data/dpt2020.csv',sep=";")

In [2]:
data.head()

,sexe,preusuel,annais,dpt,nombre
0,1,_PRENOMS_RARES,1900,02,7
1,1,_PRENOMS_RARES,1900,04,9
2,1,_PRENOMS_RARES,1900,05,8
3,1,_PRENOMS_RARES,1900,06,23
4,1,_PRENOMS_RARES,1900,07,9


In [3]:
data.isna().sum()

sexe        0
preusuel    1
annais      0
dpt         0
nombre      0
dtype: int64

In [4]:
print("Columns of dataset")
data.columns

Columns of dataset


Index(['sexe', 'preusuel', 'annais', 'dpt', 'nombre'], dtype='str')

In [5]:
print("Number of unique names")
data["preusuel"].nunique()

Number of unique names


35010

In [6]:
print("Number of row with unknown name and departments")
sum((data["annais"] == "XXXX") & (data["dpt"] == "XX"))

Number of row with unknown name and departments


37244

In [7]:
df = data[(data["annais"] != "XXXX") & (data["dpt"] != "XX")]

In [8]:
df.head()

,sexe,preusuel,annais,dpt,nombre
0,1,_PRENOMS_RARES,1900,02,7
1,1,_PRENOMS_RARES,1900,04,9
2,1,_PRENOMS_RARES,1900,05,8
3,1,_PRENOMS_RARES,1900,06,23
4,1,_PRENOMS_RARES,1900,07,9


## Scatter plot

In [9]:
df_year = (df.groupby(["preusuel", "sexe", "annais"])["nombre"].sum().reset_index())

In [10]:
df_year.head()

,preusuel,sexe,annais,nombre
0,AADIL,1,1983,3
1,AADIL,1,1992,3
2,AAHIL,1,2016,3
3,AALIYA,2,2017,3
4,AALIYAH,2,2001,9


In [11]:
df_year["Popularité annuelle"] = (
    df_year.groupby(["annais", "sexe"])["nombre"]
    .rank(ascending=False, method="min")
    .astype(int)
)
df_year.head()


,preusuel,sexe,annais,nombre,Popularité annuelle
0,AADIL,1,1983,3,869
1,AADIL,1,1992,3,1018
2,AAHIL,1,2016,3,1687
3,AALIYA,2,2017,3,1773
4,AALIYAH,2,2001,9,969


In [12]:
df_name = (
    df_year.groupby(["preusuel", "sexe"])["nombre"]
    .sum()
    .reset_index()
    .sort_values("nombre", ascending=False)
)
df_name.head()

,preusuel,sexe,nombre
9694,MARIE,2,2231903
6685,JEAN,1,1912848
12389,PIERRE,1,891170
16054,_PRENOMS_RARES,2,853451
10703,MICHEL,1,818001


In [13]:
df_name["Popularité Cumulée"] = (
    df_name.groupby("sexe")["nombre"]
    .rank(ascending=False, method="min")
    .astype(int)
)
df_name.head()

,preusuel,sexe,nombre,Popularité Cumulée
9694,MARIE,2,2231903,1
6685,JEAN,1,1912848,1
12389,PIERRE,1,891170,2
16054,_PRENOMS_RARES,2,853451,2
10703,MICHEL,1,818001,3


In [14]:
yearly_max_change = (
    df_year.sort_values(["preusuel", "sexe", "annais"])
    .groupby(["preusuel","sexe"])["nombre"]
    .diff().abs()
    .groupby([df_year["preusuel"], df_year["sexe"]]).max()
    .rename("Changement Max en 1 an")
    .reset_index()
)

max_change = (
    df_year.sort_values(["preusuel", "sexe", "annais"])
    .groupby(["preusuel","sexe"])["nombre"]
    .agg(lambda x: (x.max() - x.min())/x.max())
    .rename("Changement relatif maximum")
    .reset_index()
)

df_name = df_name.merge(yearly_max_change, on=["preusuel","sexe"], how="left")
df_name = df_name.merge(max_change, on=["preusuel","sexe"], how="left")

In [15]:
df_name.head()

,preusuel,sexe,nombre,Popularité Cumulée,Changement Max en 1 an,Changement relatif maximum
0,MARIE,2,2231903,1,15563.0,0.988303
1,JEAN,1,1912848,1,12789.0,0.989251
2,PIERRE,1,891170,2,6984.0,0.976449
3,_PRENOMS_RARES,2,853451,2,1773.0,0.955647
4,MICHEL,1,818001,3,10879.0,0.999540


In [16]:
# Popularité max
popularite_max = (
    df_year
    .groupby(["preusuel","sexe"])["Popularité annuelle"].min()
    .rename("Popularité max")
    .reset_index()
)

df_name = df_name.merge(popularite_max, on=["preusuel","sexe"], how="left")

# Popularité min
popularite_min = (
    df_year
    .groupby(["preusuel","sexe"])["Popularité annuelle"].min()
    .rename("Popularité min")
    .reset_index()
)

df_name = df_name.merge(popularite_min, on=["preusuel","sexe"], how="left")

df_name.head()

,preusuel,sexe,nombre,Popularité Cumulée,Changement Max en 1 an,Changement relatif maximum,Popularité max,Popularité min
0,MARIE,2,2231903,1,15563.0,0.988303,1,1
1,JEAN,1,1912848,1,12789.0,0.989251,1,1
2,PIERRE,1,891170,2,6984.0,0.976449,2,2
3,_PRENOMS_RARES,2,853451,2,1773.0,0.955647,1,1
4,MICHEL,1,818001,3,10879.0,0.999540,2,2


In [17]:
top10_count = (
    df_year[df_year["Popularité annuelle"] <= 10]
    .groupby(["preusuel", "sexe"])
    .size()
    .rename("Années top 10")
    .reset_index()
)

df_name = df_name.merge(top10_count, on=["preusuel", "sexe"], how="left")
df_name["Années top 10"] = df_name["Années top 10"].fillna(0).astype(int)


In [18]:
top100_count = (
    df_year[df_year["Popularité annuelle"] <= 100]
    .groupby(["preusuel", "sexe"])
    .size()
    .rename("Années top 100")
    .reset_index()
)

df_name = df_name.merge(top100_count, on=["preusuel", "sexe"], how="left")
df_name["Années top 100"] = df_name["Années top 100"].fillna(0).astype(int)


In [19]:
df_name[df_name["preusuel"]=="MARIE"]

,preusuel,sexe,nombre,Popularité Cumulée,Changement Max en 1 an,Changement relatif maximum,Popularité max,Popularité min,Années top 10,Années top 100
0,MARIE,2,2231903,1,15563.0,0.988303,1,1,91,121
493,MARIE,1,24169,222,221.0,0.997113,38,38,0,41


In [20]:
alt.Chart(df_name).mark_point().encode(
    x=alt.X("Popularité max:O", title="Popularité max", sort="descending"),
    y=alt.Y("Années top 100:Q", title="Années top 100"),
    # y=alt.Y("Changement relatif maximum:Q", title="Changement relatif maximum"),
    color=alt.Color("sexe:N", scale=alt.Scale(domain=[1, 2],
                    range=["blue", "pink"]), 
                    legend=alt.Legend(labelExpr="datum.label == '1' ? 'H' : 'F'")),
    tooltip=["preusuel", "sexe", "nombre"]
).transform_window(
    row_number="row_number()"
).transform_filter(
    "datum.row_number < 100"
)

alt.Chart(...)

In [21]:
df_name.columns

Index(['preusuel', 'sexe', 'nombre', 'Popularité Cumulée',
       'Changement Max en 1 an', 'Changement relatif maximum',
       'Popularité max', 'Popularité min', 'Années top 10', 'Années top 100'],
      dtype='str')

In [22]:
columns = ['sexe', 'nombre', 'Popularité Cumulée',
       'Changement Max en 1 an', 'Changement relatif maximum',
       'Popularité max', 'Popularité min', 'Années top 10', 'Années top 100']

x_param = alt.param(name="x_param", value="Popularité Cumulée",
                    bind=alt.binding_select(options=columns, name="X: "))
y_param = alt.param(name="y_param", value="nombre",
                    bind=alt.binding_select(options=columns, name="Y: "))

alt.Chart(df_name.head(100)).mark_point().encode(
    x=alt.X("x_val:Q"),
    y=alt.Y("y_val:Q"),
    color=alt.Color("sexe:N", scale=alt.Scale(domain=[1, 2], range=["blue", "pink"]),
                    legend=alt.Legend(labelExpr="datum.label == '1' ? 'H' : 'F'")),
    tooltip=["preusuel", "sexe", "nombre"]
).transform_calculate(
    x_val="datum[x_param]",
    y_val="datum[y_param]"
).add_params(x_param, y_param)

alt.Chart(...)

In [23]:
names = ["MARIE", "JEAN", "PIERRE", "MICHEL", "SOPHIE"]

name_param = alt.param(
    name = "name_param",
    value="MARIE",
    bind=alt.binding_select(options=names, name="Prénom: ")
)

alt.Chart(df_year[df_year["preusuel"].isin(names)]).mark_line().encode(
    x=alt.X("annais", title="Year"),
    y=alt.Y("nombre:Q", title="Count"),
    color=alt.Color("sexe:N", scale=alt.Scale(domain=[1, 2], range=["blue", "pink"]),
                    legend=alt.Legend(labelExpr="datum.label == '1' ? 'H' : 'F'")),
    tooltip=["preusuel", "annais", "nombre"]
).transform_filter(
    "datum.preusuel == name_param"
).add_params(name_param)

alt.Chart(...)

In [32]:
top_names = df_name.head(500)["preusuel"].tolist()
df_year_top = df_year[df_year["preusuel"].isin(top_names)]

columns = ['sexe', 'nombre', 'Popularité Cumulée',
       'Changement Max en 1 an', 'Changement relatif maximum',
       'Popularité max', 'Popularité min', 'Années top 10', 'Années top 100']

x_param = alt.param(name="x_param", value="Popularité Cumulée",
                    bind=alt.binding_select(options=columns, name="X: "))
y_param = alt.param(name="y_param", value="nombre",
                    bind=alt.binding_select(options=columns, name="Y: "))

brush = alt.selection_point(fields=["preusuel", "sexe"], toggle=True, empty=True)

scatter = alt.Chart(df_name.head(500)).mark_point(size=60).encode(
    x=alt.X("x_val:Q", title='', sort="descending"),
    y=alt.Y("y_val:Q", title=''),
    color=alt.Color("sexe:N", scale=alt.Scale(domain=[1, 2], range=["blue", "pink"]),
                    legend=alt.Legend(labelExpr="datum.label == '1' ? 'H' : 'F'")),
    opacity=alt.condition(brush, alt.value(1.0), alt.value(0.15)),
    tooltip=["preusuel", "sexe", "nombre"]
).transform_calculate(
    x_val="datum[x_param]",
    y_val="datum[y_param]"
).add_params(x_param, y_param, brush).properties(width=800, height=300)

lines = alt.Chart(df_year_top).mark_line().encode(
    x=alt.X("annais:O", title="Année", axis=alt.Axis(labelExpr="parseInt(datum.value) % 5 == 0 ? datum.value : ''",
                      tickCount=len(df_year_top["annais"].unique()))),
    y=alt.Y("nombre:Q", title="Nombre"),
    detail="preusuel:N",
    color=alt.Color("sexe:N", scale=alt.Scale(domain=[1, 2], range=["blue", "pink"]), legend=None),
    tooltip=["preusuel", "annais", "nombre"]
).transform_filter(brush).properties(width=800, height=300)

lines & scatter

alt.VConcatChart(...)